In [ ]:
# Cell 1 - imports and paths
from pathlib import Path
import yaml
import pandas as pd

ROOT = Path.cwd()

# If the notebook is inside notebooks/, move up one level to repo root
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

BASE_DIR = ROOT / "runs" / "segment"
REPORT_DIR = ROOT / "reports"
REPORT_DIR.mkdir(exist_ok=True)

print("ROOT:", ROOT)
print("BASE_DIR:", BASE_DIR)
print("BASE_DIR exists:", BASE_DIR.exists())

In [ ]:
# Cell 2 - list of runs to inventory
RUNS = [
    "car_defect_detection/model4_control_no_multiscale_7cls",
    "car_defect_detection/model4_control_no_multiscale_7cls_unfreeze",
    "car_defect_detection/yolo26m_1024_stage2_clean_multiscale-3",
    "car_defect_detection/yolo26m_1024_stage2_clean_multiscale-4",
    "car_defect_detection/yolo26m_1024_stage2_clean_multiscale_v2",
    "model5_resume_adapt_7cls/stage1_head_warmup_7cls",
    "model5_resume_adapt_7cls/stage2_differential_finetune_7cls",
    "model5_resume_adapt/stage1_head_warmup-2",
    "model5_resume_adapt/stage1_head_warmup-3",
    "model5_resume_adapt/stage2_differential_finetune",
    "model5_resume_adapt/stage2_differential_finetune-3",
    "model5_resume_adapt/stage2_differential_finetune-4",
    "model5_resume_adapt/stage2_differential_finetune-5",
    "stage1_head_warmup_7cls_extended/stage1_head_warmup_7cls_extended",
]

for run in RUNS:
    run_path = BASE_DIR / run

    print(
        run,
        "| exists:", run_path.exists(),
        "| results.csv:", (run_path / "results.csv").exists(),
        "| best.pt:", (run_path / "weights" / "best.pt").exists(),
        "| last.pt:", (run_path / "weights" / "last.pt").exists(),
    )

In [ ]:
# Cell 3 - helper functions

def load_args(run_path):
    yaml_path = run_path / "args.yaml"

    if not yaml_path.exists():
        return {"args_error": "args.yaml not found"}

    try:
        with open(yaml_path, "r") as f:
            args = yaml.safe_load(f) or {}
    except Exception as exc:
        return {"args_error": str(exc)}

    return {
        "project": args.get("project"),
        "name": args.get("name"),
        "model_file": Path(str(args.get("model", ""))).name,
        "data_file": Path(str(args.get("data", ""))).name,
        "imgsz": args.get("imgsz"),
        "epochs_config": args.get("epochs"),
        "batch": args.get("batch"),
        "lr0": args.get("lr0"),
        "freeze": args.get("freeze"),
        "multi_scale": args.get("multi_scale"),
        "mosaic": args.get("mosaic"),
        "scale_aug": args.get("scale"),
        "degrees": args.get("degrees"),
        "patience": args.get("patience"),
    }


def load_metrics(run_path):
    csv_path = run_path / "results.csv"

    if not csv_path.exists():
        return {"metrics_error": "results.csv not found"}

    try:
        df = pd.read_csv(csv_path)
        df.columns = [c.strip() for c in df.columns]

        required_cols = [
            "epoch",
            "metrics/mAP50(M)",
            "metrics/mAP50-95(M)",
            "metrics/precision(M)",
            "metrics/recall(M)",
        ]

        missing_cols = [c for c in required_cols if c not in df.columns]

        if missing_cols:
            return {"metrics_error": f"missing columns: {missing_cols}"}

        for col in required_cols:
            df[col] = pd.to_numeric(df[col], errors="coerce")

        df = df.dropna(subset=["epoch", "metrics/mAP50(M)"])

        if df.empty:
            return {"metrics_error": "no valid metric rows"}

        best_row = df.loc[df["metrics/mAP50(M)"].idxmax()]
        final_row = df.iloc[-1]

        return {
            "total_epochs_trained": int(len(df)),
            "best_epoch": int(best_row["epoch"]),
            "best_mask_map50": round(float(best_row["metrics/mAP50(M)"]), 3),
            "best_mask_map50_95": round(float(best_row["metrics/mAP50-95(M)"]), 3),
            "best_mask_precision": round(float(best_row["metrics/precision(M)"]), 3),
            "best_mask_recall": round(float(best_row["metrics/recall(M)"]), 3),
            "final_epoch": int(final_row["epoch"]),
            "final_mask_map50": round(float(final_row["metrics/mAP50(M)"]), 3),
        }

    except Exception as exc:
        return {"metrics_error": str(exc)}

In [ ]:
# Cell 4 - build inventory table

records = []

for run in RUNS:
    run_path = BASE_DIR / run

    record = {
        "run_folder": run,
        "exists": run_path.exists(),
    }

    if record["exists"]:
        record.update(load_args(run_path))
        record.update(load_metrics(run_path))

        record["has_best_pt"] = (run_path / "weights" / "best.pt").exists()
        record["has_last_pt"] = (run_path / "weights" / "last.pt").exists()

        # Rough schema guess based only on folder name.
        # We will verify manually before the report.
        if "7cls" in run:
            record["class_schema_guess"] = "7-class"
        elif "model5_resume_adapt/" in run:
            record["class_schema_guess"] = "8-class?"
        else:
            record["class_schema_guess"] = "verify"

    records.append(record)

df = pd.DataFrame(records)

if "best_mask_map50" in df.columns:
    df = df.sort_values(
        by="best_mask_map50",
        ascending=False,
        na_position="last"
    )

output_csv = REPORT_DIR / "experiment_inventory.csv"
df.to_csv(output_csv, index=False)

print("Saved inventory CSV to:", output_csv)

In [ ]:
# Cell 5 - display readable benchmark table

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

display_cols = [
    "run_folder",
    "class_schema_guess",
    "name",
    "imgsz",
    "epochs_config",
    "freeze",
    "multi_scale",
    "mosaic",
    "total_epochs_trained",
    "best_epoch",
    "best_mask_map50",
    "best_mask_map50_95",
    "best_mask_precision",
    "best_mask_recall",
    "final_mask_map50",
    "has_best_pt",
]

existing_cols = [c for c in display_cols if c in df.columns]

df[existing_cols]

In [ ]:
# Cell 6 - Gather report assets for the 6 selected models

import shutil

# 1. Define the 6 selected models and their report-friendly names
REPORT_MODELS = {
    "car_defect_detection/model4_control_no_multiscale_7cls": "1_baseline_freeze_7cls",
    "car_defect_detection/model4_control_no_multiscale_7cls_unfreeze": "2_baseline_unfreeze_7cls",
    "model5_resume_adapt/stage2_differential_finetune-5": "3_model5_stage2_8cls",
    "model5_resume_adapt_7cls/stage1_head_warmup_7cls": "4_model5_stage1_7cls_15ep",
    "model5_resume_adapt_7cls/stage2_differential_finetune_7cls": "5_model5_stage2_7cls_forgetting",
    "stage1_head_warmup_7cls_extended/stage1_head_warmup_7cls_extended": "6_champion_extended_7cls"
}

# 2. Define the visual assets we want to collect
TARGET_IMAGES = [
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "MaskPR_curve.png",
    "MaskF1_curve.png",
    "MaskP_curve.png",
    "MaskR_curve.png",
    "results.png"  # Contains the loss/mAP training curves
]

# 3. Setup the destination directory
ASSETS_DIR = ROOT / "reports" / "assets"
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

# 4. Gather and copy the images
for run_path_str, report_name in REPORT_MODELS.items():
    src_dir = BASE_DIR / run_path_str
    dst_dir = ASSETS_DIR / report_name
    dst_dir.mkdir(parents=True, exist_ok=True)
    
    copied_files = []
    missing_files = []
    
    for img_name in TARGET_IMAGES:
        src_file = src_dir / img_name
        if src_file.exists():
            shutil.copy2(src_file, dst_dir / img_name)
            copied_files.append(img_name)
        else:
            missing_files.append(img_name)
            
    print(f"✅ [{report_name}] Copied {len(copied_files)} images. Missing: {', '.join(missing_files) if missing_files else 'None'}")

print(f"\n✨ All assets gathered in: {ASSETS_DIR.relative_to(ROOT)}")

In [ ]:
# Cell 7 - Evaluate selected models on the Test dataset

from ultralytics import YOLO
import pandas as pd

# 1. Define exact dataset paths based on your find/cat output
DATASET_7CLS = "/home/lamacpp/Documents/car_defect_detection/data/processed/yolo_seg_clean_2200_7cls/dataset.yaml"
DATASET_8CLS = "/home/lamacpp/Documents/car_defect_detection/data/processed/yolo_seg_clean_2200_/dataset.yaml"

# 2. Define the 6 selected models, their schema, and report names
TEST_MODELS = [
    ("car_defect_detection/model4_control_no_multiscale_7cls", "7-class", "1_baseline_freeze_7cls"),
    ("car_defect_detection/model4_control_no_multiscale_7cls_unfreeze", "7-class", "2_baseline_unfreeze_7cls"),
    ("model5_resume_adapt/stage2_differential_finetune-5", "8-class", "3_model5_stage2_8cls"),
    ("model5_resume_adapt_7cls/stage1_head_warmup_7cls", "7-class", "4_model5_stage1_7cls_15ep"),
    ("model5_resume_adapt_7cls/stage2_differential_finetune_7cls", "7-class", "5_model5_stage2_7cls_forgetting"),
    ("stage1_head_warmup_7cls_extended/stage1_head_warmup_7cls_extended", "7-class", "6_champion_extended_7cls")
]

test_results = []

print("Starting Test Set Evaluation on RTX 4090...\n")

for run_path_str, schema, report_name in TEST_MODELS:
    weight_path = BASE_DIR / run_path_str / "weights" / "best.pt"
    
    if not weight_path.exists():
        print(f"⚠️ Skipping {report_name}: best.pt not found at {weight_path}")
        continue
        
    dataset_path = DATASET_7CLS if schema == "7-class" else DATASET_8CLS
    
    print(f"🔄 Evaluating {report_name} ({schema}) on TEST set...")
    
    try:
        # Load model
        model = YOLO(str(weight_path))
        
        # Run validation on test split
        # device=0 uses the RTX 4090, verbose=False keeps the notebook clean
        results = model.val(
            data=dataset_path, 
            split="test", 
            imgsz=1024, 
            batch=4, 
            device=0, 
            verbose=False
        )
        
        # Extract mask metrics
        # We use results_dict but add fallbacks to direct attributes just in case
        metrics_dict = results.results_dict
        
        test_mask_map50 = metrics_dict.get("metrics/mAP50(M)", getattr(results.seg, "map", 0.0))
        test_mask_map50_95 = metrics_dict.get("metrics/mAP50-95(M)", getattr(results.seg, "map95", 0.0))
        test_mask_precision = metrics_dict.get("metrics/precision(M)", getattr(results.seg, "mp", 0.0))
        test_mask_recall = metrics_dict.get("metrics/recall(M)", getattr(results.seg, "mr", 0.0))
        
        record = {
            "report_name": report_name,
            "schema": schema,
            "test_mask_map50": round(float(test_mask_map50), 3),
            "test_mask_map50_95": round(float(test_mask_map50_95), 3),
            "test_mask_precision": round(float(test_mask_precision), 3),
            "test_mask_recall": round(float(test_mask_recall), 3),
        }
        test_results.append(record)
        
        print(f"✅ {report_name} -> Test Mask mAP50: {test_mask_map50:.3f}\n")
        
    except Exception as e:
        print(f"❌ Error evaluating {report_name}: {e}\n")

# 3. Create DataFrame and display
df_test = pd.DataFrame(test_results)
print("\n" + "="*80)
print("FINAL TEST SET BENCHMARKS")
print("="*80)

# Display nicely in Jupyter
if not df_test.empty:
    display(df_test)

    # Save to CSV
    test_csv_path = REPORT_DIR / "test_benchmarks.csv"
    df_test.to_csv(test_csv_path, index=False)
    print(f"\nSaved test benchmarks to: {test_csv_path}")
else:
    print("No test results were generated.")